In [2]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, attn_implementation="eager")  # eager is needed to return attention weights
model.eval()

# 1. Layers and heads per layer
n_layers = model.config.num_hidden_layers
n_heads = model.config.num_attention_heads
print("Transformer layers:", n_layers)
print("Attention heads per layer:", n_heads)

# 2. Total attention heads
print("Total attention heads:", n_layers * n_heads)

# 3. Attention weights: layer 3, head 2
sentence = "Transformers use self-attention to understand language."
inputs = tokenizer(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

# outputs.attentions[layer] has shape (batch, heads, seq_len, seq_len)
# indices are 0-based: layer 3 -> 2, head 2 -> 1
attention = outputs.attentions[2][0, 1]

print("Tokens:", tokens)
print("Attention shape:", attention.shape)
print("Attention weights:\n", attention)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Transformer layers: 12
Attention heads per layer: 12
Total attention heads: 144
Tokens: ['[CLS]', 'transformers', 'use', 'self', '-', 'attention', 'to', 'understand', 'language', '.', '[SEP]']
Attention shape: torch.Size([11, 11])
Attention weights:
 tensor([[7.3176e-01, 7.9132e-03, 1.0591e-02, 5.5538e-03, 1.6532e-02, 6.8604e-03,
         1.0085e-02, 1.1588e-02, 5.5202e-03, 3.8613e-02, 1.5499e-01],
        [8.8883e-01, 4.7016e-03, 3.4382e-02, 1.7840e-04, 2.7377e-03, 5.0542e-03,
         9.0185e-04, 7.3192e-04, 4.0114e-04, 5.1106e-03, 5.6975e-02],
        [1.4273e-01, 5.6892e-01, 5.4059e-03, 3.5340e-02, 4.1156e-02, 6.7886e-02,
         8.9602e-02, 3.2741e-04, 5.3299e-03, 8.9671e-03, 3.4334e-02],
        [3.3953e-01, 7.3164e-02, 1.5250e-03, 1.6106e-04, 2.0160e-01, 8.1124e-02,
         6.3570e-02, 2.5709e-02, 1.6392e-03, 2.8494e-02, 1.8348e-01],
        [1.4291e-01, 6.9677e-02, 7.1289e-02, 6.3943e-01, 1.2104e-02, 1.2819e-02,
         3.4381e-03, 2.5649e-03, 1.4182e-02, 9.2673e-03, 2.2316e

In [ ]:
"""Rows and columns of the attention matrix

Rows are the query tokens, the token that is “looking” at the others.
Columns are the key tokens, the tokens being looked at.
Entry [i, j] is how much token i attends to token j in this head.
Each row is a softmax output, so it sums to 1. Columns do not.
Row i is the distribution over all tokens that token i uses to build its new representation.
BERT is bidirectional with no causal mask, so every token can attend to every other token, including itself, [CLS] and [SEP]."""